# Cost Engineering: Token Accounting and Dashboards

**Phase 07** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-07/06-cost-engineering.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

Your LLM feature launches. Three months later your infrastructure bill has a line item: $4,200 last month for AI API calls. Your CTO asks which feature drives the most cost. You open your code. Every team made API calls directly. Nobody tracked tokens. Nobody tracked which endpoint called which model. You have no answer.

You start guessing: "It's probably the summarization feature, that uses a lot of tokens." You are wrong. It's the search-intent classifier that runs on every keystroke because a developer set it to fire on `onChange` instead of `onBlur`. You would have caught this in week one if you had cost accounting. Instead you found it in month three, after spending $12,600.

Cost engi...

## The Concept

### Where LLM Costs Come From

```
+---------------------------+-------------------+---------------------------+
| Cost Source               | Typical Impact    | Fix                       |
+---------------------------+-------------------+---------------------------+
| Long system prompts       | 30-60% of input   | Prompt caching (L07)      |
|   repeated on every call  | tokens per call   |                           |
+---------------------------+-------------------+---------------------------+
| Verbose model outputs     | 2-5x output cost  | Explicit length            |
|   (no max length...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
L06: Cost Engineering - Token Accounting and Dashboards
Phase 07 - Observability

Demonstrates:
- Per-request cost computation from API usage fields
- SQLite-backed cost store with model, feature, and user dimensions
- ASCII cost breakdown report
- Monthly projection and budget alert
- CostAccounting high-level interface
"""

import sqlite3
from dataclasses import dataclass
from datetime import date, datetime, timezone
from typing import Optional

import anthropic

### Pricing Table (USD per 1M tokens, 2026)

In [ ]:
PRICING: dict[str, dict[str, float]] = {
    "claude-3-5-haiku-20241022": {
        "input": 0.80,
        "output": 4.00,
        "cache_write": 1.00,
        "cache_read": 0.08,
    },
    "claude-3-5-sonnet-20241022": {
        "input": 3.00,
        "output": 15.00,
        "cache_write": 3.75,
        "cache_read": 0.30,
    },
    "claude-opus-4-5": {
        "input": 15.00,
        "output": 75.00,
        "cache_write": 18.75,
        "cache_read": 1.50,
    },
}

# Default to Haiku pricing for unknown models
_DEFAULT_MODEL = "claude-3-5-haiku-20241022"


def compute_cost(
    model: str,
    input_tokens: int,
    output_tokens: int,
    cache_write_tokens: int = 0,
    cache_read_tokens: int = 0,
) -> float:
    """
    Compute USD cost for a single API call.

    Returns cost in dollars. Typical Haiku call: $0.0000003 - $0.000003.
    Output tokens cost 5x more per token than input tokens.
    """
    prices = PRICING.get(model, PRICING[_DEFAULT_MODEL])
    cost = (
        (input_tokens * prices["input"])
        + (output_tokens * prices["output"])
        + (cache_write_tokens * prices["cache_write"])
        + (cache_read_tokens * prices["cache_read"])
    ) / 1_000_000
    return round(cost, 8)

### SQLite Store

In [ ]:
_CREATE_SQL = """
CREATE TABLE IF NOT EXISTS llm_costs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    ts TEXT NOT NULL,
    model TEXT NOT NULL,
    feature_name TEXT NOT NULL DEFAULT 'unknown',
    user_id TEXT,
    input_tokens INTEGER NOT NULL DEFAULT 0,
    output_tokens INTEGER NOT NULL DEFAULT 0,
    cache_write_tokens INTEGER NOT NULL DEFAULT 0,
    cache_read_tokens INTEGER NOT NULL DEFAULT 0,
    cost_usd REAL NOT NULL DEFAULT 0.0,
    latency_ms REAL
);
"""


def init_db(db_path: str = "llm_costs.db") -> sqlite3.Connection:
    """Create the database and cost table if they do not exist."""
    conn = sqlite3.connect(db_path)
    conn.execute(_CREATE_SQL)
    conn.commit()
    return conn


def record_cost(
    conn: sqlite3.Connection,
    model: str,
    input_tokens: int,
    output_tokens: int,
    feature_name: str = "unknown",
    user_id: Optional[str] = None,
    cache_write_tokens: int = 0,
    cache_read_tokens: int = 0,
    latency_ms: Optional[float] = None,
) -> float:
    """
    Persist a single API call's cost dimensions.
    Returns the computed cost_usd.
    """
    cost = compute_cost(
        model, input_tokens, output_tokens, cache_write_tokens, cache_read_tokens
    )
    conn.execute(
        """INSERT INTO llm_costs
           (ts, model, feature_name, user_id, input_tokens, output_tokens,
            cache_write_tokens, cache_read_tokens, cost_usd, latency_ms)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)""",
        (
            datetime.now(timezone.utc).isoformat(),
            model,
            feature_name,
            user_id,
            input_tokens,
            output_tokens,
            cache_write_tokens,
            cache_read_tokens,
            cost,
            latency_ms,
        ),
    )
    conn.commit()
    return cost

### Cost Breakdown Report

In [ ]:
def cost_report(conn: sqlite3.Connection) -> str:
    """
    Render an ASCII cost breakdown report to stdout.
    Shows totals, breakdown by model, by feature, and top 5 users.
    """
    lines: list[str] = []
    lines.append("=" * 62)
    lines.append("LLM COST REPORT")
    lines.append("=" * 62)

    # Totals
    row = conn.execute(
        "SELECT SUM(cost_usd), SUM(input_tokens), SUM(output_tokens), COUNT(*) FROM llm_costs"
    ).fetchone()
    total_cost = row[0] or 0.0
    total_in = row[1] or 0
    total_out = row[2] or 0
    total_calls = row[3] or 0

    lines.append(f"\n{'Total calls':<20}: {total_calls:,}")
    lines.append(f"{'Total cost':<20}: ${total_cost:.4f}")
    lines.append(f"{'Input tokens':<20}: {total_in:,}")
    lines.append(f"{'Output tokens':<20}: {total_out:,}")
    if total_calls > 0:
        lines.append(f"{'Avg cost/call':<20}: ${total_cost/total_calls:.6f}")

    # By model
    lines.append("\n--- By Model ---")
    lines.append(f"{'Model':<35} {'Calls':>6} {'Cost ($)':>10} {'% Total':>8}")
    lines.append("-" * 62)
    for model, calls, cost in conn.execute(
        "SELECT model, COUNT(*), SUM(cost_usd) FROM llm_costs "
        "GROUP BY model ORDER BY SUM(cost_usd) DESC"
    ):
        pct = (cost / total_cost * 100) if total_cost else 0.0
        lines.append(f"{model:<35} {calls:>6} {cost:>10.4f} {pct:>7.1f}%")

    # By feature
    lines.append("\n--- By Feature ---")
    lines.append(
        f"{'Feature':<25} {'Calls':>6} {'Cost ($)':>10} {'Avg/call':>10} {'Avg Out Tokens':>15}"
    )
    lines.append("-" * 70)
    for feat, calls, cost, avg_out in conn.execute(
        "SELECT feature_name, COUNT(*), SUM(cost_usd), AVG(output_tokens) "
        "FROM llm_costs GROUP BY feature_name ORDER BY SUM(cost_usd) DESC"
    ):
        avg = cost / calls if calls else 0.0
        lines.append(f"{feat:<25} {calls:>6} {cost:>10.4f} {avg:>10.6f} {avg_out:>15.0f}")

    # Top 5 users
    lines.append("\n--- Top 5 Users by Cost ---")
    lines.append(f"{'User ID':<25} {'Calls':>6} {'Cost ($)':>10} {'Avg/call':>10}")
    lines.append("-" * 55)
    for uid, calls, cost in conn.execute(
        "SELECT COALESCE(user_id, 'anonymous'), COUNT(*), SUM(cost_usd) "
        "FROM llm_costs GROUP BY user_id ORDER BY SUM(cost_usd) DESC LIMIT 5"
    ):
        avg = cost / calls if calls else 0.0
        lines.append(f"{str(uid):<25} {calls:>6} {cost:>10.4f} {avg:>10.6f}")

    lines.append("\n" + "=" * 62)
    return "\n".join(lines)

### Budget Alert

In [ ]:
def monthly_projection(conn: sqlite3.Connection) -> float:
    """
    Project current-month spend to end-of-month based on daily run rate.
    Uses 30-day month approximation.
    """
    today = date.today()
    month_start = today.replace(day=1).isoformat()
    days_elapsed = today.day

    row = conn.execute(
        "SELECT SUM(cost_usd) FROM llm_costs WHERE ts >= ?",
        (month_start,),
    ).fetchone()
    cost_so_far = row[0] or 0.0

    if days_elapsed == 0:
        return 0.0
    return round(cost_so_far / days_elapsed * 30, 4)


def check_budget_alert(
    conn: sqlite3.Connection,
    monthly_budget_usd: float,
    alert_threshold: float = 0.8,
) -> dict:
    """
    Return budget alert status.

    alert_threshold: fraction of budget that triggers an alert (default 0.8 = 80%).
    """
    projection = monthly_projection(conn)
    ratio = projection / monthly_budget_usd if monthly_budget_usd > 0 else 0.0
    alert = ratio >= alert_threshold

    return {
        "projected_monthly_usd": projection,
        "budget_usd": monthly_budget_usd,
        "utilization_pct": round(ratio * 100, 1),
        "alert": alert,
        "message": (
            f"ALERT: Projected ${projection:.2f} is {ratio*100:.0f}% "
            f"of ${monthly_budget_usd:.2f} budget"
            if alert
            else f"OK: Projected ${projection:.2f} ({ratio*100:.0f}% of ${monthly_budget_usd:.2f})"
        ),
    }

### High-Level Interface

In [ ]:
class CostAccounting:
    """
    Drop-in cost tracking for any LLM-calling service.

    Usage:
        accounting = CostAccounting()

        # In your LLM wrapper, after each API call:
        cost = accounting.track(
            model=response.model,
            input_tokens=response.usage.input_tokens,
            output_tokens=response.usage.output_tokens,
            feature_name="search_intent_classifier",
            user_id=current_user.id,
        )
    """

    def __init__(self, db_path: str = "llm_costs.db"):
        self.db_path = db_path
        self.conn = init_db(db_path)

    def track(
        self,
        model: str,
        input_tokens: int,
        output_tokens: int,
        feature_name: str = "unknown",
        user_id: Optional[str] = None,
        cache_write_tokens: int = 0,
        cache_read_tokens: int = 0,
        latency_ms: Optional[float] = None,
    ) -> float:
        """Record a call. Returns cost in USD."""
        return record_cost(
            self.conn,
            model=model,
            input_tokens=input_tokens,
            output_tokens=output_tokens,
            feature_name=feature_name,
            user_id=user_id,
            cache_write_tokens=cache_write_tokens,
            cache_read_tokens=cache_read_tokens,
            latency_ms=latency_ms,
        )

    def report(self) -> str:
        """Return the ASCII cost breakdown report."""
        return cost_report(self.conn)

    def budget_alert(
        self, monthly_budget_usd: float, alert_threshold: float = 0.8
    ) -> dict:
        """Return budget alert status dict."""
        return check_budget_alert(self.conn, monthly_budget_usd, alert_threshold)

### API wrapper with cost tracking (optional - requires ANTHROPIC_API_KEY)

In [ ]:
import time as _time


def call_with_cost_tracking(
    prompt: str,
    feature_name: str = "unknown",
    user_id: Optional[str] = None,
    accounting: Optional[CostAccounting] = None,
    model: str = "claude-3-5-haiku-20241022",
) -> tuple[str, float]:
    """
    Make a Claude API call and record cost.
    Returns (response_text, cost_usd).
    """
    if accounting is None:
        accounting = CostAccounting()

    client = anthropic.Anthropic()
    start = _time.monotonic()

    response = client.messages.create(
        model=model,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    latency_ms = (_time.monotonic() - start) * 1000

    cost = accounting.track(
        model=model,
        input_tokens=response.usage.input_tokens,
        output_tokens=response.usage.output_tokens,
        cache_read_tokens=getattr(response.usage, "cache_read_input_tokens", 0),
        cache_write_tokens=getattr(response.usage, "cache_creation_input_tokens", 0),
        feature_name=feature_name,
        user_id=user_id,
        latency_ms=latency_ms,
    )

    text = response.content[0].text if response.content else ""
    return text, cost

### Demo

In [ ]:
def demo() -> None:
    """
    Seed a demo database with synthetic call data and render the report.
    No API key needed for this demo.
    """
    import random
    import os

    db_path = "/tmp/llm_costs_demo.db"
    if os.path.exists(db_path):
        os.remove(db_path)

    accounting = CostAccounting(db_path=db_path)

    # Synthetic call data
    features = ["search_intent", "summarize", "classify", "extract_entities"]
    models = [
        "claude-3-5-haiku-20241022",
        "claude-3-5-haiku-20241022",
        "claude-3-5-haiku-20241022",  # most calls on haiku
        "claude-3-5-sonnet-20241022",  # some on sonnet
    ]
    users = ["user_001", "user_002", "user_003", None]

    random.seed(42)
    for _ in range(50):
        model = random.choice(models)
        feature = random.choice(features)
        user = random.choice(users)
        input_tok = random.randint(50, 800)
        output_tok = random.randint(20, 500)
        accounting.track(
            model=model,
            input_tokens=input_tok,
            output_tokens=output_tok,
            feature_name=feature,
            user_id=user,
            latency_ms=random.uniform(200, 1500),
        )

    print(accounting.report())

    alert = accounting.budget_alert(monthly_budget_usd=0.05)
    print(f"\nBudget check: {alert['message']}")

    # Verify cost formula
    cost = compute_cost("claude-3-5-haiku-20241022", 100, 50)
    expected = (100 * 0.80 + 50 * 4.00) / 1_000_000
    assert abs(cost - expected) < 1e-10, f"Cost formula wrong: {cost} != {expected}"
    print("\nCost formula verification: PASS")

### Demo

In [ ]:
demo()

---

## Self-Check

Answer these without running code first:

**1. What is the cost impact of this switch, and what is the right engineering response?**

- A. Accept the switch - accuracy matters more than cost for production systems.
- B. Reject the switch - Opus is about 19x more expensive per input token and likely adds no accuracy for a simple binary classification task; measure Opus accuracy first before spending more.
- C. Accept only if the team agrees to cap monthly spend at the current amount.
- D. Switch to Opus for 10% of traffic as an A/B test and compare costs monthly.

**2. What is the correct cost engineering response?**

- A. Switch the search_intent_classifier to a smaller model to reduce per-call cost.
- B. Add prompt caching to the search_intent_classifier's system prompt.
- C. Fix the call frequency first: debounce the trigger to fire on pause-in-typing or onBlur, then measure the new call volume before optimizing per-call cost.
- D. Cache the classifier's output so identical queries do not call the model.

**3. Should you add cache token fields to your schema now or later?**

- A. Add them now - even if unused today, adding them later requires a schema migration and gaps in historical data.
- B. Add them only when you implement prompt caching - premature schema design adds complexity.
- C. Cache tokens are included in input_tokens, so no separate field is needed.
- D. Cache tokens do not affect cost, so tracking them adds no value.

**4. What is wrong with the current projection, and how would you fix it?**

- A. Nothing is wrong - the projection correctly reflects the run rate at day 5.
- B. The projection assumes constant daily spend, so a front-loaded batch job inflates it. Add a flag to exclude batch job costs from the steady-state projection.
- C. Reduce the alert threshold from 80% to 50% to give more warning time.
- D. Change the projection to use median daily cost instead of mean daily cost.

**5. What is the most actionable interpretation of this data?**

- A. Block or throttle the CEO's account to reduce costs.
- B. This is expected for a power user - no action needed.
- C. Investigate whether the CEO's usage pattern reveals a missing feature (e.g., they are doing something manually that should be automated) and whether per-user rate limits or semantic caching would help.
- D. The data is not meaningful because the CEO has special access patterns.

**6. What is the most cost-effective fix?**

- A. Switch to a cheaper model for marketing copy generation.
- B. Add explicit output length instructions to the prompt: 'Write a subject line and 2-3 sentence email body. Be concise - total response under 150 words.'
- C. Truncate the model's output to 150 tokens after generation.
- D. Use streaming and cut off the stream at 150 tokens.

**7. What is the correct architecture for this dashboard?**

- A. Run the cost aggregation query on every page load - SQLite is fast enough.
- B. Cache the dashboard data and refresh it on a schedule (e.g., every 5 minutes or hourly) - real-time accuracy is not needed for cost reporting.
- C. Build a streaming pipeline that updates cost totals in memory on every API call.
- D. Use a separate analytics database (ClickHouse) even at $500/month scale.

_Answers are in `checks.json` in the lesson directory._